# Bab 3 — Hierarchical Clustering

Notebook pendamping Bab 3 Level 4.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

- Menjelaskan *agglomerative* (bottom-up) dan *divisive* (top-down).
- Membaca dendrogram dan memotongnya untuk mendapat jumlah kelompok.
- Menjelaskan empat linkage: single, complete, average, Ward.
- Memakai `AgglomerativeClustering` dengan `n_clusters`, `linkage`, `metric`, `distance_threshold`.
- Menjelaskan biaya komputasi hierarchical clustering.

## Konsep singkat

Hierarchical clustering membangun pohon penggabungan. Dua kelompok terdekat digabung berulang sampai tinggal satu, lalu pohon (dendrogram) dipotong untuk memilih jumlah kelompok. **Linkage** adalah aturan jarak antar dua kelompok. Scikit-learn hanya menyediakan agglomerative lewat `AgglomerativeClustering`. Dendrogram digambar dengan SciPy.

## Dataset

Lima titik 1D (hitungan manual), `make_blobs`, `make_moons`, data dua gerombol dengan jembatan, dan Mall Customers. `random_state=42`.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform

## Bagian B: Lima titik dan matriks jarak

Hasilnya sama dengan tabel manual di halaman bab.

In [ ]:
# Lima titik pada satu garis (satu fitur), diberi nama A sampai E
X_line = np.array([[0.0], [1.0], [3.5], [7.0], [9.0]])
names = ["A", "B", "C", "D", "E"]

# pdist menghitung jarak semua pasangan, squareform mengubahnya menjadi matriks persegi
dist_matrix = squareform(pdist(X_line))
print(pd.DataFrame(dist_matrix, index=names, columns=names))

## Bagian C: Matriks linkage

Kolom: nomor kelompok 1, nomor kelompok 2, jarak penggabungan (tinggi), ukuran kelompok hasil. Nomor 0-4 adalah titik A-E, nomor 5 dan seterusnya adalah kelompok baru.

In [ ]:
for method in ["single", "complete", "average", "ward"]:
    Z = linkage(X_line, method=method)
    print(f"\nLinkage: {method}")
    print(pd.DataFrame(Z, columns=["klaster_1", "klaster_2", "jarak_gabung", "ukuran"]).round(3).to_string())

In [ ]:
# Rumus Ward (tinggi gabung di SciPy): sqrt(2 * nA * nB / (nA + nB)) * jarak antar-rata-rata
A = X_line[[0, 1, 2]]   # klaster {A, B, C}
B = X_line[[3, 4]]      # klaster {D, E}
nA, nB = len(A), len(B)
ward_height = np.sqrt(2 * nA * nB / (nA + nB)) * np.linalg.norm(A.mean(axis=0) - B.mean(axis=0))
print(f"Rata-rata A,B,C = {A.mean():.3f}; rata-rata D,E = {B.mean():.3f}")
print(f"Tinggi gabung Ward terakhir (rumus) = {ward_height:.3f}")

## Bagian D: Dendrogram kecil

Urutan penggabungan sama pada keempat linkage, tetapi tingginya berbeda.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=False)
for ax, method in zip(axes, ["single", "complete", "average", "ward"]):
    Z = linkage(X_line, method=method)
    dendrogram(Z, labels=names, ax=ax, color_threshold=0, above_threshold_color="#0072B2")
    ax.set_title(f"Linkage: {method}")
    ax.set_xlabel("Titik")
axes[0].set_ylabel("Jarak saat digabung")
plt.tight_layout()
plt.show()

## Bagian E: `AgglomerativeClustering` pada data blobs

`children_` berisi pasangan yang digabung. Nomor di bawah 150 adalah titik asli.

In [ ]:
X, _ = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=42)

agg = AgglomerativeClustering(n_clusters=3, linkage="ward", compute_distances=True)
labels = agg.fit_predict(X)

print("Ukuran kelompok:", np.bincount(labels).tolist())
print("n_clusters_ :", agg.n_clusters_)
print("n_leaves_   :", agg.n_leaves_)
print("children_ (3 baris pertama, pasangan yang digabung):")
print(agg.children_[:3])
print("distances_ (3 pertama dan 2 terakhir):", agg.distances_[:3].round(3).tolist(), agg.distances_[-2:].round(3).tolist())

In [ ]:
for threshold in [1, 3, 8, 30]:
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=threshold, linkage="ward").fit(X)
    print(f"distance_threshold = {threshold:<3} -> n_clusters_ = {model.n_clusters_}")

## Bagian F: Linkage pada bentuk data yang berbeda

Single linkage mengikuti bulan sabit, tetapi gagal pada data dengan jembatan titik (chaining).

In [ ]:
X_moons, _ = make_moons(n_samples=200, noise=0.06, random_state=42)

# Dua gerombol bulat yang "dijembatani" enam titik di antaranya
X_two, _ = make_blobs(n_samples=150, centers=[[-3, 0], [3, 0]], cluster_std=0.7, random_state=42)
bridge = np.column_stack([np.linspace(-1.2, 1.2, 6), np.zeros(6)])
X_bridge = np.vstack([X_two, bridge])

datasets = {"moons": X_moons, "bridge": X_bridge}
linkages = ["single", "complete", "average", "ward"]

shape_labels = {}
for data_name, data in datasets.items():
    for method in linkages:
        lab = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(data)
        shape_labels[(data_name, method)] = lab
        print(f"{data_name:<7} {method:<9} ukuran kelompok: {np.bincount(lab).tolist()}")

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for row, (data_name, data) in enumerate(datasets.items()):
    for col, method in enumerate(linkages):
        ax = axes[row, col]
        lab = shape_labels[(data_name, method)]
        for k in sorted(set(lab)):
            ax.scatter(data[lab == k, 0], data[lab == k, 1], s=14, color=palette[k % len(palette)])
        if row == 0:
            ax.set_title(f"Linkage: {method}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[row, 0].set_ylabel(data_name)
plt.tight_layout()
plt.show()

## Bagian G: Dendrogram Mall Customers

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])

Z_mall = linkage(X_mall, method="ward")
print("Bentuk matriks linkage:", Z_mall.shape, "(n - 1 baris, 4 kolom)")
print("Tinggi gabung 5 langkah terakhir:", Z_mall[-5:, 2].round(2).tolist())

In [ ]:
cut_height = 7.0
fig, ax = plt.subplots(figsize=(10, 4.5))
dendrogram(Z_mall, truncate_mode="lastp", p=16, ax=ax, color_threshold=cut_height,
           above_threshold_color="#444444", show_leaf_counts=True)
ax.axhline(cut_height, color="#D55E00", linestyle="--", label=f"Garis potong (tinggi = {cut_height})")
ax.set_xlabel("Cabang (angka dalam kurung = jumlah pelanggan)")
ax.set_ylabel("Jarak Ward saat digabung")
ax.set_title("Dendrogram Mall Customers (dipangkas ke 16 cabang terakhir)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for height in [3, 5, 7, 10, 20]:
    cut_labels = fcluster(Z_mall, t=height, criterion="distance")
    print(f"Potong di tinggi {height:<2} -> {len(set(cut_labels))} kelompok")

agg5 = AgglomerativeClustering(n_clusters=5, linkage="ward")
df["Cluster"] = agg5.fit_predict(X_mall)
print("\nUkuran 5 kelompok (urut):", sorted(np.bincount(df["Cluster"]).tolist()))
print(df.groupby("Cluster")[features].mean().round(1))

## Bagian H: Parameter `metric` dan biaya memori

Biaya memori matriks jarak naik kuadratik terhadap jumlah data. Angkanya perkiraan, bukan pengukuran.

In [ ]:
for metric in ["euclidean", "manhattan", "cosine"]:
    model = AgglomerativeClustering(n_clusters=5, linkage="average", metric=metric).fit(X_mall)
    print(f"linkage=average, metric={metric:<9} -> ukuran kelompok: {sorted(np.bincount(model.labels_).tolist())}")

In [ ]:
print(f"{'n (jumlah data)':>16} {'pasangan n(n-1)/2':>20} {'memori (GiB, float64)':>22}")
for n in [200, 10_000, 100_000]:
    pairs = n * (n - 1) // 2
    gib = pairs * 8 / 1024**3
    print(f"{n:>16,} {pairs:>20,} {gib:>22.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for k in range(5):
    members = df[df["Cluster"] == k]
    ax.scatter(members[features[0]], members[features[1]], s=22,
               color=palette[k % len(palette)], label=f"Kelompok {k}")
ax.set_xlabel(features[0])
ax.set_ylabel(features[1])
ax.set_title("Agglomerative (Ward, 5 kelompok) pada Mall Customers")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Interpretasi

- Tinggi penggabungan terakhir pada lima titik: single 3,5; complete 9; average 6,5; Ward 10,07.
- Pada moons, hanya single linkage mengikuti bentuk bulan sabit. Pada data berjembatan, single linkage gagal (155 dan 1).
- Dendrogram Mall Customers: pemotongan di antara tinggi 4,35 dan 9,45 menghasilkan 5 kelompok.
- Memori matriks jarak untuk 100.000 titik sekitar 37 GiB.

## Latihan

1. Hitung manual linkage single, complete, average untuk titik 0, 2, 5, 11, lalu cocokkan dengan `linkage`.
2. Jalankan blobs dengan `n_clusters=2` dan `4`.
3. Coba `distance_threshold` 60, 100, dan 130. Kaitkan dengan `distances_`.
4. Naikkan `noise` moons ke 0,2. Bagaimana single linkage dan Ward?
5. Jalankan Ward pada Mall Customers dengan 3, 4, 6 kelompok.
6. Hitung perkiraan memori matriks jarak untuk 20.000 titik.

Jawaban ada di halaman bab.

## Ringkasan

- Hierarchical clustering membangun pohon, jumlah kelompok dipilih sesudahnya dengan memotong dendrogram.
- Linkage: single (terdekat), complete (terjauh), average (rata-rata), Ward (kenaikan WCSS).
- Single mengikuti bentuk memanjang tetapi rentan terhadap chaining dan noise. Ward cocok untuk gerombol bulat.
- Biaya memori dan waktu naik kuadratik, dan estimator ini tidak punya `predict`.